In [407]:
import pandas as pd
import numpy as np
import warnings

In [408]:
warnings.filterwarnings('ignore')

In [409]:
df = pd.read_csv('../data/raw/visualisation_df.csv').drop(columns=['Unnamed: 0', 'X', 'probability_of_failure'])
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,assessment,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment,one_year_prob
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,Satisfactory,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN,0.014321
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,Not Available,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0,0.008131
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,Satisfactory,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN,0.015885
3,SOAD00862,Navaldia,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Rated,295.1,25.3,NaN,NaN,0,NaN,NaN,NaN,0.016265
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,Not Rated,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0,0.011027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20801,SOAD16536,Lyndrassia,No,Flood Risk Reduction,Rockfill,35.872,1.667,7411000.0,480.62681,84247.85257,...,Not Available,820.9,510.6,NaN,47.0,0,47.0,2.0,47.0,0.009078
20802,SOAD13145,Lyndrassia,No,Hydroelectric,Gravity,141.296,0.963,375000.0,405.40236,62491.43656,...,Not Available,850.2,176.6,62.9,52.0,0,52.0,5.0,52.0,0.008579
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,Not Available,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0,0.010708
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,Not Available,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0,0.007160


In [410]:
df = df.loc[df['primary_type'] == 'Earth']

In [411]:
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,assessment,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment,one_year_prob
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,Satisfactory,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN,0.014321
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,Not Available,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0,0.008131
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,Satisfactory,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN,0.015885
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,Not Rated,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0,0.011027
5,SOAD02227,Lyndrassia,Yes,Recreation,Earth,14.332,0.127,NaN,0.06107,NaN,...,Unsatisfactory,11.5,733.7,8.2,27.0,0,27.0,27.0,27.0,0.014654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20798,SOAD11040,Navaldia,No,Hydroelectric,Earth,3.956,0.105,NaN,1499.77873,50184.37347,...,Satisfactory,850.1,163.8,83.8,59.0,0,59.0,3.0,2.0,0.010512
20799,SOAD12695,Navaldia,No,Hydroelectric,Earth,3.444,0.230,NaN,1539.68108,49808.84958,...,Satisfactory,770.7,331.4,87.1,59.0,0,59.0,3.0,2.0,0.008308
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,Not Available,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0,0.010708
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,Not Available,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0,0.007160


In [412]:
df['damage_loss'] = df['damage_loss'].fillna(0)
df['dam_repair_loss'] = df['dam_repair_loss'].fillna(0)
df['business_interruption_loss'] = df['business_interruption_loss'].fillna(0)

In [413]:
df['business_interruption_loss'].isna().sum()

0

In [414]:
df['total_loss'] = df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss']
df['expected_loss'] = df['one_year_prob'] * (df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss'])

### hazard rating factor

In [415]:
def calculate_hazard_rating_factor(df, hazard):
    curr_hazard_mean = (df[df['hazard'] == hazard].describe()['total_loss'].loc['50%'] + df[df['hazard'] == hazard].describe()['total_loss'].loc['50%']) / 2
    low_hazard_mean = (df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%'] + df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%']) / 2
    hazard_rating_factor = curr_hazard_mean/low_hazard_mean
    return hazard_rating_factor

In [416]:
hazard_low_rf = calculate_hazard_rating_factor(df, 'Low')
hazard_high_rf = calculate_hazard_rating_factor(df, 'High')
hazard_significant_rf = calculate_hazard_rating_factor(df, 'Significant')
hazard_undetermined_rf = calculate_hazard_rating_factor(df, 'Undetermined')

In [417]:
# Define mapping dictionary
hazard_mapping = {
    'Low': hazard_low_rf,
    'High': hazard_high_rf,
    'Significant': hazard_significant_rf,
    'Undetermined': hazard_undetermined_rf
}

# Create a new column using map()
df['hazard_rating_factor'] = df['hazard'].map(hazard_mapping)


In [418]:
df['region'].value_counts()

Navaldia      8374
Lyndrassia    7920
Flumevale     3074
Name: region, dtype: int64

### Regulation rating factor

In [419]:
df['no_BI_loss'] = df['dam_repair_loss'] + df['damage_loss']

In [420]:
df['w'] = df['no_BI_loss'] / df['total_loss']
df['w'] = df['w'].fillna(1)
df['failure_rate'] = df['one_year_prob'] * df['w']
def calculate_regulation_rating_factor(df, region):
    regulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'Yes')]['failure_rate'])
    unregulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'No')]['failure_rate'])
    regulated_rating_factor = unregulated_region_failure_rate / regulated_region_failure_rate 
    return regulated_rating_factor

In [421]:
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,years_from_modification,years_from_inspection,years_from_assessment,one_year_prob,total_loss,expected_loss,hazard_rating_factor,no_BI_loss,w,failure_rate
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,NaN,10.0,NaN,0.014321,325.8,4.665852,3.970874,317.7,0.975138,0.013965
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,98.0,7.0,98.0,0.008131,1658.0,13.481425,7.224272,1658.0,1.000000,0.008131
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.015885,791.4,12.571001,3.970874,782.8,0.989133,0.015712
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,44.0,4.0,44.0,0.011027,220.1,2.426972,7.224272,214.5,0.974557,0.010746
5,SOAD02227,Lyndrassia,Yes,Recreation,Earth,14.332,0.127,NaN,0.06107,NaN,...,27.0,27.0,27.0,0.014654,753.4,11.040423,7.224272,745.2,0.989116,0.014495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20798,SOAD11040,Navaldia,No,Hydroelectric,Earth,3.956,0.105,NaN,1499.77873,50184.37347,...,59.0,3.0,2.0,0.010512,1097.7,11.539411,7.224272,1013.9,0.923659,0.009710
20799,SOAD12695,Navaldia,No,Hydroelectric,Earth,3.444,0.230,NaN,1539.68108,49808.84958,...,59.0,3.0,2.0,0.008308,1189.2,9.879331,7.224272,1102.1,0.926757,0.007699
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,71.0,3.0,71.0,0.010708,1413.0,15.130069,7.224272,1413.0,1.000000,0.010708
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,60.0,4.0,60.0,0.007160,1256.1,8.993341,7.224272,1256.1,1.000000,0.007160


In [422]:
regulated_mapping_1 = {
    'Yes': 1
}
df['regulated_rating_factor'] = df['regulated_dam'].map(regulated_mapping_1)

In [423]:
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,years_from_inspection,years_from_assessment,one_year_prob,total_loss,expected_loss,hazard_rating_factor,no_BI_loss,w,failure_rate,regulated_rating_factor
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,10.0,NaN,0.014321,325.8,4.665852,3.970874,317.7,0.975138,0.013965,1.0
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,7.0,98.0,0.008131,1658.0,13.481425,7.224272,1658.0,1.000000,0.008131,NaN
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.015885,791.4,12.571001,3.970874,782.8,0.989133,0.015712,1.0
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,4.0,44.0,0.011027,220.1,2.426972,7.224272,214.5,0.974557,0.010746,1.0
5,SOAD02227,Lyndrassia,Yes,Recreation,Earth,14.332,0.127,NaN,0.06107,NaN,...,27.0,27.0,0.014654,753.4,11.040423,7.224272,745.2,0.989116,0.014495,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20798,SOAD11040,Navaldia,No,Hydroelectric,Earth,3.956,0.105,NaN,1499.77873,50184.37347,...,3.0,2.0,0.010512,1097.7,11.539411,7.224272,1013.9,0.923659,0.009710,NaN
20799,SOAD12695,Navaldia,No,Hydroelectric,Earth,3.444,0.230,NaN,1539.68108,49808.84958,...,3.0,2.0,0.008308,1189.2,9.879331,7.224272,1102.1,0.926757,0.007699,NaN
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,3.0,71.0,0.010708,1413.0,15.130069,7.224272,1413.0,1.000000,0.010708,NaN
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,4.0,60.0,0.007160,1256.1,8.993341,7.224272,1256.1,1.000000,0.007160,NaN


In [424]:
# Define mapping dictionary
regulated_mapping = {
    'Navaldia': calculate_regulation_rating_factor(df, 'Navaldia'),
    'Lyndrassia': calculate_regulation_rating_factor(df, 'Lyndrassia'),
    'Flumevale': calculate_regulation_rating_factor(df, 'Flumevale')
}

# Create a new column using map()
df['unregulated_rating_factor'] = df['region'].map(regulated_mapping)


In [425]:
df['regulated_rating_factor'] = df['regulated_rating_factor'].fillna(df['unregulated_rating_factor'])

In [426]:
df['region'].value_counts()

Navaldia      8374
Lyndrassia    7920
Flumevale     3074
Name: region, dtype: int64

### GDP rating factor

In [427]:
gdp_df = pd.read_excel('../data/raw/soaGDP.xlsx', sheet_name='2025 Nominal GDP')
pop_df = pd.read_csv('../data/raw/population10yrs.csv')

In [428]:
pop_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2019,45363514,7067855,39808697,92240066
1,2020,45502051,7097789,40175188,92775028
2,2021,45651175,7131024,40565887,93348086
3,2022,45599000,7157446,40953108,93709554
4,2023,45311937,7239138,42148205,94699280
5,2024,45161092,7267582,42526941,95150152
6,2025,45145328,7324559,43313896,95925972
7,2026,45259214,7353108,43693348,96378123
8,2027,45404777,7409970,44479504,97152512
9,2028,45473424,7438623,44859669,97605929


In [429]:
gdp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2025,4.671259e+06,534401.257379,3.779710e+06,8.985371e+06


In [430]:
for feature in gdp_df.columns:
    gdp_df[feature] = gdp_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)
    pop_df[feature] = pop_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)

In [431]:
gdp_pp_df = pd.DataFrame({
    'Year': gdp_df['Year']
})
# change here for gdp in different year
for feature in gdp_df.drop(columns=['Year']).columns:
    gdp_pp_df[feature] = gdp_df[feature].item()/pop_df.iloc[6][feature]

In [432]:
gdp_pp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2025,0.103472,0.07296,0.087263,0.09367


In [433]:
for feature in gdp_pp_df.drop(columns=['Year', 'Tarrodan']):
    gdp_pp_df[feature] = gdp_pp_df['Tarrodan']/gdp_pp_df[feature]

In [434]:
regulated_mapping = {
    'Navaldia': gdp_pp_df['Navaldia'].item(),
    'Lyndrassia': gdp_pp_df['Lyndrassia'].item(),
    'Flumevale': gdp_pp_df['Flumevale'].item()
}

# Create a new column using map()
df['gdp_rating_factor'] = df['region'].map(regulated_mapping)


### summary

In [435]:
df['total_rating_factor'] = df['hazard_rating_factor'] * df['regulated_rating_factor'] * df['gdp_rating_factor']

In [436]:
df[['hazard_rating_factor', 'regulated_rating_factor', 'gdp_rating_factor', 'total_rating_factor']].describe()

,hazard_rating_factor,regulated_rating_factor,gdp_rating_factor,total_rating_factor
count,19368.000000,19368.000000,19368.000000,19368.000000
mean,2.791123,1.029417,1.132780,3.292715
std,2.677251,0.301665,0.138102,3.516583
min,0.095631,0.074239,0.905271,0.058796
25%,1.000000,1.000000,1.073417,1.073417
50%,1.000000,1.000000,1.073417,1.884630
75%,3.970874,1.000000,1.283849,5.098003
max,7.224272,1.467953,1.283849,13.615079


In [437]:
def log_rating_factor(x):
    if x > 2:
        x = np.log(x)
        return x
    else:
        return x
    
df['total_rating_factor'] = df['total_rating_factor'].apply(log_rating_factor)

In [438]:
df['total_rating_factor'].describe()

count    19368.000000
mean         1.442688
std          0.583101
min          0.058796
25%          0.905271
50%          1.283849
75%          1.884630
max          2.611178
Name: total_rating_factor, dtype: float64

### rating factor table

In [439]:
rating_factor_df = df.groupby(['region', 'regulated_dam', 'hazard'])['total_rating_factor'].value_counts()

In [440]:
# Reset index if needed for better CSV structure
df_reset = rating_factor_df.reset_index(name='count')

# Export to CSV
df_reset.to_excel('../plot/rating_factor_df.xlsx', index=False)


### premium

In [441]:
df['dam_repair_loss_owner'] = df['dam_repair_loss'] * 0.8
df['dam_repair_loss_business'] = df['dam_repair_loss'] * 0.1
df['dam_repair_loss_people'] = df['dam_repair_loss'] * 0.1

In [442]:
df['TP_loss_owner'] = df['damage_loss'] * 0.6
df['TP_loss_business'] = df['damage_loss'] * 0.2
df['TP_loss_people'] = df['damage_loss'] * 0.2

In [443]:
df['BI_loss_owner'] = df['business_interruption_loss'] * 0.6
df['BI_loss_business'] = df['business_interruption_loss'] * 0.4

In [444]:
df['loss_owner'] = df['dam_repair_loss_owner'] + df['TP_loss_owner'] + df['BI_loss_owner']
df['loss_business'] = df['dam_repair_loss_business'] + df['TP_loss_business'] + df['BI_loss_business']
df['loss_people'] = df['dam_repair_loss_people'] + df['TP_loss_people']

In [ ]:
for part in ['owner', 'business', 'people']:
    df[f'premium_{part}'] = df[f'loss_{part}'] * df['one_year_prob']

In [446]:
df[['premium_owner', 'premium_business', 'premium_people']].describe()

,premium_owner,premium_business,premium_people
count,19368.000000,19368.000000,19368.000000
mean,2.635691,0.695339,0.655189
std,2.932330,0.725589,0.710673
min,0.000000,0.000000,0.000000
25%,0.425520,0.095567,0.068219
50%,1.507329,0.442502,0.406946
75%,3.906344,1.064611,1.019183
max,22.393721,5.090902,4.815006


In [447]:
for part in ['owner', 'business', 'people']:
    df[f'adjusted_premium_{part}'] = df[f'premium_{part}'] * df['total_rating_factor']

In [448]:
df[['adjusted_premium_owner', 'adjusted_premium_business', 'adjusted_premium_people']].describe()

,adjusted_premium_owner,adjusted_premium_business,adjusted_premium_people
count,19368.000000,19368.000000,19368.000000
mean,4.451995,1.184839,1.129513
std,5.730891,1.460819,1.425451
min,0.000000,0.000000,0.000000
25%,0.459567,0.107287,0.080343
50%,1.974511,0.548842,0.507788
75%,6.412665,1.803078,1.718694
max,50.438477,11.051306,10.515024


### owner premium

In [449]:
owner_BI_premium = []
for region in df['region'].value_counts().keys():
    owner_BI_premium.append(df[(df['region'] == region) & (df['business_interruption_loss'] > 0)]['premium_owner'].sum()/np.square(((df['region'] == region) & (df['business_interruption_loss'] > 0)).sum()))

In [450]:
owner_no_BI_premium = []
for region in df['region'].value_counts().keys():
    owner_no_BI_premium.append(df[(df['region'] == region) & (df['business_interruption_loss'] == 0)]['premium_owner'].sum()/np.square(((df['region'] == region) & (df['business_interruption_loss'] == 0)).sum()))

In [451]:
owner_BI_premium

[0.000676435758606065, 0.0010320128678863606, 0.0018438544189826613]

In [452]:
owner_no_BI_premium

[0.0005582321554773182, 0.0003294384182032021, 0.004061325130985245]

### BI pool

In [453]:
df.groupby('region')['premium_business'].sum()

region
Flumevale     298.835562
Lyndrassia    501.261930
Navaldia      546.635980
Name: premium_business, dtype: float64

### person premium

In [454]:
population_near_dam_ratio_df = pd.DataFrame([[0.17, 0.06, 0.45]], columns=df['region'].value_counts().keys())

In [455]:
population_near_dam_ratio_df

,Navaldia,Lyndrassia,Flumevale
0,0.17,0.06,0.45


In [ ]:
person_premium = []
for region in df['region'].value_counts().keys():
    person_premium.append(df[(df['region'] == region)]['premium_people'].sum()/(pop_df[feature][6] * population_near_dam_ratio_df[feature]).item())

In [464]:
person_premium

[6.972357326954321e-05, 6.55641915100644e-05, 3.704781888012883e-05]

### final pool

In [467]:
(df.groupby('region')[['adjusted_premium_owner', 'adjusted_premium_business', 'adjusted_premium_people']].sum().T.sum()/10) / (df.groupby('region')[['adjusted_premium_owner', 'adjusted_premium_business', 'adjusted_premium_people']].sum().T.sum()/10).sum()

region
Flumevale     0.192097
Lyndrassia    0.401071
Navaldia      0.406832
dtype: float64